## Plusieurs sections du notebook reprennent des parties de code issues du dépôt GitHub : https://github.com/ageron/handson-ml2

In [ ]:
import os
import pandas as pd
import numpy as np

# On fixe la graine aléatoire
np.random.seed(42)

# Projet ML bout-en-bout
## Prédire le prix des maisons en Californie à partir de données sur la population, le revenu moyen, le prix médian des maisons par quartier

## Quadrage du problème
La première question à se poser et à quoi va servir ce modèle. Dans certains cas, un modèle de prédiction automatique peut être l'objectif finale mais dans d'autres il peut être une étape intermédiaire. Connaître l'objectif va permettre de sélectionner le type d'algorithme de Machine Learning à évaluer. Dans le cas présent ce sera un modèle de régression.

## Obtenir de la donnée

In [ ]:
datapath = os.path.join("../datasets", "housing", "housing.csv")
housing = pd.read_csv(datapath)
housing.head(10)

In [ ]:
housing.info()

In [ ]:
housing["ocean_proximity"].value_counts()

In [ ]:
housing.describe()

In [ ]:
housing[['longitude']].apply(lambda x: np.quantile(x, 0.1))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
housing.hist(bins=50, figsize=(20,15))
plt.show()

In [ ]:
housing.shape

## Suppression des valeurs plafonnées (artéfacts)

En observant `describe()` et les histogrammes, deux variables présentent une valeur maximale « plafonnée » (*capping*) qui regroupe artificiellement de nombreux quartiers sur une seule valeur :
- `median_house_value` est plafonné à **500 001 \$** (965 quartiers) ;
- `housing_median_age` est plafonné à **52 ans** (1 103 quartiers).

Ces plateaux ne reflètent pas la réalité et risquent de perturber l'apprentissage des modèles. On les affiche d'abord pour les repérer, puis on supprime les lignes concernées.

In [ ]:
housing.loc[housing['median_house_value']>= 500001]

In [ ]:
housing = housing.loc[housing['median_house_value'] < 500001]

In [ ]:
housing.loc[housing['housing_median_age']>=52.0]

In [ ]:
housing = housing.loc[housing['housing_median_age'] < 52.0]

## Visualisation de la donnée

In [ ]:
import matplotlib.image as mpimg
california_img=mpimg.imread(os.path.join("../datasets", "housing", "california.png"))
ax = housing.plot(kind="scatter", x="longitude", y="latitude", figsize=(10,7),
                  s=housing['population']/100, label="Population",
                  c="median_house_value", cmap=plt.get_cmap("jet"),
                  colorbar=False, alpha=0.4)
plt.imshow(california_img, extent=[-124.55, -113.80, 32.45, 42.05], alpha=0.5,
           cmap=plt.get_cmap("jet"))
plt.ylabel("Latitude", fontsize=14)
plt.xlabel("Longitude", fontsize=14)

prices = housing["median_house_value"]
tick_values = np.linspace(prices.min(), prices.max(), 11)
cbar = plt.colorbar(ticks=tick_values/prices.max())
cbar.ax.set_yticklabels(["$%dk"%(round(v/1000)) for v in tick_values], fontsize=14)
cbar.set_label('Median House Value', fontsize=16)

plt.legend(fontsize=16)
plt.show()

## Analyse exploratoire multivariée : recherche de corrélation
Corrélation de Pearson : https://en.wikipedia.org/wiki/Pearson_correlation_coefficient

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_matrix

In [ ]:
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
from pandas.plotting import scatter_matrix
attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]
scatter_matrix(housing[attributes], figsize=(12, 8))

In [ ]:
housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.1)

## Feature engineering
Combinaison de caractéristiques plus informatives

In [ ]:
housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]  # nombre de pièces par maison
housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]  # nombre de chambres par rapport au pièces
housing["population_per_household"]=housing["population"]/housing["households"]  # nombre de personnes par maison

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].abs().sort_values(ascending=False)

## Préparation de la donnée pour les algorithmes de Machine Learning

### Gestion des valeurs manquantes
La très grande majorité des algorithmes ne peuvent fonctionner avec des valeurs manquantes.
3 options possibles pour gérer les valeurs manquantes :
- Supprimer les exemples ayant des valeurs
- Supprimer la colonne complètement
- Remplacer les valeurs manquantes (par zéro, la moyenne, la médiane, la veleur la plus fréquente, etc.)

### option 1 : supprimer les lignes ayant des nan
```python
housing.dropna(subset=["total_bedrooms"])  
```

### option 2 : supprimer la colonne
```python
housing.drop("total_bedrooms", axis=1)
```

### L'option 3 : Remplacer les nan. Attention pour cette méthode, il faut calculer la valeur médiane sur les données d'apprentissage et l'appliquer sur tous les autres datasets
```python
median = housing["total_bedrooms"].median()
housing["total_bedrooms"].fillna(median, inplace=True)
```

### L'option 3 est la technique dite d'<I>imputing</I>. Trouver une règle métier afin de trouver des valeurs de remplacements.

In [ ]:
housing.isna().mean()

In [ ]:
# Option 1 choisie
print(housing.shape[0])  # Avant
housing = housing.dropna()
print(housing.shape[0])  # Après. Suppression de ~200 quartiers

### Gestion des données catégorielles
Les algorithmes de Machine Learning ne peuvent gérer les données catégorielles. Le one hot encoding (<a href="https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html">```OneHotEncoder```</a>) est l'une des meilleurs façon de transformer les données catégorielles en données numériques.
Son principe consiste à créer une colonne binaire par catégorie.
Une autre façon plus simple est de rattacher une valeur par catégorie mais cela a le défaut de rapprocher artificiellement des catégories ayant des valeurs proches. Cette méthode est le ordinal encoder : <a href="https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html">```OrdinalEncoder```</a>.

In [ ]:
housing["ocean_proximity"]

In [ ]:
housing["ocean_proximity"].value_counts()

In [ ]:
# Méthode à utiliser dans le cas où les catégories peuvent être ordonnées : Ordinal Encoding
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
ordinal_encoder.fit_transform(housing[["ocean_proximity"]])

In [ ]:
ordinal_encoder.categories_

In [ ]:
# Méthode du one hot enconding est celle qui sera utilisée pour ce projet
from sklearn.preprocessing import OneHotEncoder

one_hot_encoder = OneHotEncoder(sparse_output=False)
housing_cat_1hot = one_hot_encoder.fit_transform(housing[["ocean_proximity"]])
housing_cat_1hot

In [ ]:
one_hot_encoder.categories_

#### A savoir que le one-hot-encoding est pratique quand le nombre de catégories est limité. Dans le cas où le nombre est grand (traitement du texte), il existe des techniques appellées <i>embedding</i>

## Mise à l'échelle des données (Feature Scaling)
L'une des transformations les plus importantes est la mise à l'échelle. Dans de rares cas, les algorithmes de Machine Learning ne fonctionne pas avec des données ayant des valeurs ayant des plages très différentes.
Dans la plupart des cas, remettre à l'échelle les labels n'est pas requis.
Les deux mises à l'échelle les plus connues sont :
- <i>Min-Max scaling</i> (appelé également <i>Normalization</i>) : $$ z = \frac{x-min(x)}{max(x)-min(x)} $$
- <i>Standardization</i> : $$ z = \frac{x-\mu}{\sigma} $$

In [ ]:
housing.dtypes

In [ ]:
housing.select_dtypes(include=np.float64)

## La mise à l'échelle nécessite de faire le split train/test

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(housing, train_size=0.8, random_state=42) 

In [ ]:
# Min-Max Scaling
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()
z = min_max_scaler.fit_transform(housing.select_dtypes(include=np.float64))
pd.DataFrame(data=z, columns = train_set.select_dtypes(include=np.float64).columns).describe()

In [ ]:
# Standardization
from sklearn.preprocessing import StandardScaler

stand_scaler = StandardScaler()
z = stand_scaler.fit_transform(train_set.select_dtypes(include=np.float64))
pd.DataFrame(data=z, columns = train_set.select_dtypes(include=np.float64).columns).describe()

## Apprentissage d'un algorithme ML

## Sélectionner une mesure de performance
Une métrique classique dans les problèmes de régression est à la racine de l'erreur quadratique moyenne (Root Mean Squared Error ou RMSE) :
$$ RMSE = \sqrt{\frac{1}{m}\sum_{i=1}^{m}{\Big({h(\textbf{x}^{(i)}) -y^{(i)}}\Big)^2}} $$

In [ ]:
x_train = train_set.drop("median_house_value", axis=1)
y_train = train_set["median_house_value"].copy()

x_test = test_set.drop("median_house_value", axis=1)
y_test = test_set["median_house_value"].copy()

In [ ]:
x_train

In [ ]:
# Préparation de la donnée : choix du standard scaler pour les variables numériques et du OneHot pour la variable catégorielle
x_train_prepared = stand_scaler.fit_transform(x_train.select_dtypes(include=np.float64))
x_train_prepared = np.concatenate((x_train_prepared, one_hot_encoder.fit_transform(x_train[["ocean_proximity"]])), axis=1)

# ATTENTION : Il faut utiliser .transform() sur test afin de ne pas écraser les paramètres de normalisation trouvés.
x_test_prepared = stand_scaler.transform(x_test.select_dtypes(include=np.float64))
x_test_prepared = np.concatenate((x_test_prepared, one_hot_encoder.transform(x_test[["ocean_proximity"]])), axis=1)

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(x_train_prepared, y_train)

In [ ]:
print("Predictions:", lin_reg.predict(x_train_prepared))

In [ ]:
print("Labels:", y_train.values)

In [ ]:
from sklearn.metrics import root_mean_squared_error

y_train_predictions = lin_reg.predict(x_train_prepared)
lin_rmse_train = root_mean_squared_error(y_train, y_train_predictions)
lin_rmse_train
print(f"RMSE of LinReg on training set: {lin_rmse_train:.2f}")

In [ ]:
y_test_predictions = lin_reg.predict(x_test_prepared)
lin_rmse_test = root_mean_squared_error(y_test, y_test_predictions)
print(f"RMSE of LinReg on testing set: {lin_rmse_test:.2f}")

### DummyRegressor

In [ ]:
y_pred_mean = y_train.mean()
y_pred_mean

In [ ]:
y_pred_dummy = np.ones(len(y_test)) * y_pred_mean
y_pred_dummy

In [ ]:
root_mean_squared_error(y_test, y_pred_dummy)

### Changement d'algorithme : l'arbre de décision 

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(x_train_prepared, y_train)

In [ ]:
print(f"RMSE of DT on training set: {root_mean_squared_error(y_train, tree_reg.predict(x_train_prepared)):.2f}")
print(f"RMSE of DT on testing set: {root_mean_squared_error(y_test, tree_reg.predict(x_test_prepared)):.2f}")

## Exercice : Recherche d'hyperparamètres

L'objectif est de trouver les meilleurs hyperparamètres pour le `DecisionTreeRegressor` (ou un autre algorithme si vous le souhaitez) en utilisant `GridSearchCV` et/ou `RandomizedSearchCV` de scikit-learn.

**Consignes :**
1. Définir une grille d'hyperparamètres à explorer
2. Utiliser `GridSearchCV` avec validation croisée (cv=5) et le scoring `neg_root_mean_squared_error`
3. Afficher les meilleurs hyperparamètres trouvés et le meilleur score
4. Évaluer le meilleur modèle sur le test set et comparer avec le modèle par défaut

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

param_grid = {
    "max_depth": [None, 5, 10],
    
}

grid_search = GridSearchCV / RandomizedSearchCV (
    # TODO
)
grid_search.fit(x_train_prepared, y_train)

print(f"\nMeilleurs hyperparamètres : {grid_search.best_params_}")
print(f"Meilleur RMSE (CV) : {-grid_search.best_score_:.2f}")

best_model = grid_search.best_estimator_
print(f"\nRMSE du meilleur modèle sur le test set : {root_mean_squared_error(y_test, best_model.predict(x_test_prepared)):.2f}")
print(f"RMSE du modèle par défaut sur le test set : {root_mean_squared_error(y_test, tree_reg.predict(x_test_prepared)):.2f}")

### Exercice — Comparer MAE, RMSE et R²
Sur des prédictions de régression, calculez et comparez les trois métriques. Observez l'effet d'une **valeur aberrante** sur la RMSE par rapport à la MAE.


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rng = np.random.default_rng(0)
y_true = rng.normal(300_000, 50_000, size=200)
y_pred = y_true + rng.normal(0, 20_000, size=200)   # bon modèle

def report(y, yhat, label):
    mae = mean_absolute_error(y, yhat)
    rmse = mean_squared_error(y, yhat) ** 0.5
    r2 = r2_score(y, yhat)
    print(f'{label:18} MAE={mae:10.0f}  RMSE={rmse:10.0f}  R2={r2:.3f}')

report(y_true, y_pred, 'Sans aberrante')

# On injecte UNE grosse erreur
# TODO y_pred_out = ...
# report(y_true, y_pred_out, 'Avec 1 aberrante')
# -> la RMSE explose alors que la MAE bouge peu : RMSE pénalise les grosses erreurs.